# VoltPath — EV Fleet Optimizer: Exploratory Analysis

This notebook walks through the key analytical steps behind the Streamlit dashboard.
All data is synthetic and labeled for portfolio/educational use.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#060d1a'
matplotlib.rcParams['axes.facecolor'] = '#0a1628'
matplotlib.rcParams['text.color'] = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor'] = '#e2e8f0'
matplotlib.rcParams['xtick.color'] = '#94a3b8'
matplotlib.rcParams['ytick.color'] = '#94a3b8'

In [ ]:
# Load processed data (run run_pipeline.py first)
fleet = pd.read_csv('../data/processed/fleet_operations.csv', parse_dates=['date'])
zones = pd.read_csv('../data/processed/zone_summary.csv')
chargers = pd.read_csv('../data/processed/chargers.csv')
hubs = pd.read_csv('../data/processed/recommended_hubs.csv')
print(f'Fleet: {fleet.shape}, Zones: {zones.shape}, Chargers: {chargers.shape}')

## 1. Fleet Overview Stats

In [ ]:
print('=== Fleet Summary ===')
print(f"Vehicles: {fleet['vehicle_id'].nunique()}")
print(f"Days: {fleet['date'].nunique()}")
print(f"Total Trips: {fleet['trip_count'].sum():,}")
print(f"Total Revenue: ₹{fleet['total_revenue'].sum():,.0f}")
print(f"Total Downtime Cost: ₹{fleet['downtime_cost'].sum():,.0f}")
loss_pct = fleet['downtime_cost'].sum() / fleet['total_revenue'].sum() * 100
print(f"Downtime Loss %: {loss_pct:.1f}%")

## 2. Zone Risk Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accessibility score
zones_sorted = zones.sort_values('accessibility_score')
colors = ['#fc8181' if v < 0.3 else '#f6ad55' if v < 0.6 else '#48bb78'
          for v in zones_sorted['accessibility_score']]
axes[0].barh(zones_sorted['zone'], zones_sorted['accessibility_score'], color=colors)
axes[0].set_title('Charging Accessibility Score by Zone', color='#e2e8f0')
axes[0].set_xlabel('Score (0=no access, 1=excellent)', color='#94a3b8')

# Downtime risk
zones_r = zones.sort_values('downtime_risk_score', ascending=False)
colors2 = ['#fc8181' if v > 0.7 else '#f6ad55' if v > 0.5 else '#48bb78'
           for v in zones_r['downtime_risk_score']]
axes[1].barh(zones_r['zone'], zones_r['downtime_risk_score'], color=colors2)
axes[1].set_title('Downtime Risk Score by Zone', color='#e2e8f0')
axes[1].set_xlabel('Risk Score (0=safe, 1=critical)', color='#94a3b8')

plt.tight_layout()
plt.show()

## 3. Recommended Hub Locations

In [ ]:
print('=== Recommended Charging Hubs ===')
print(hubs[['priority_rank','zone_proximity','estimated_annual_savings','lat','lon']].to_string(index=False))

## 4. ROI Simulation — 1 to 10 New Chargers

In [ ]:
from src.analysis.analysis import simulate_roi

results = []
for n in range(1, 11):
    r = simulate_roi(zones, fleet, new_chargers=n)
    results.append(r)

roi_df = pd.DataFrame(results)
print(roi_df[['new_chargers','capex_lakh','revenue_recovered_inr','annual_roi_pct','estimated_payback_months']].to_string(index=False))